# VQC Based on IQC:AIL

## Imports

In [1]:
import qiskit
from qiskit_machine_learning.algorithms import VQC
from qiskit.circuit import QuantumCircuit,Parameter
from qiskit.compiler import transpile
from qiskit_aer import Aer
from qiskit.visualization import plot_histogram, visualize_transition, plot_bloch_vector
from qiskit.circuit.library import UnitaryGate,Initialize
from qiskit.quantum_info import Statevector,partial_trace, DensityMatrix

import pennylane as qml
from pennylane import numpy as pnp
import qutip
from toqito import state_props

import numpy as np
from scipy.linalg import expm as expMatrix
from sympy.physics.quantum.dagger import Dagger
import math

from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold,train_test_split, KFold
from sklearn.multiclass import OneVsRestClassifier
from sklearn.utils.multiclass import unique_labels
from sklearn.utils.validation import check_array, check_is_fitted
from sklearn.preprocessing import MinMaxScaler
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn import preprocessing
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score, make_scorer, roc_auc_score, classification_report

from ucimlrepo import fetch_ucirepo

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import pdflatex

import pandas as pd


## Base de Dados

In [2]:
#Gerando o dataset
bc = fetch_ucirepo(id=17)
# data (as pandas dataframes) 
X_iris = bc.data.features 
X_iris = X_iris.values
y_iris = bc.data.targets
y_iris = y_iris.values

#Parâmetros
RANDOM_SEED = 42
LEARNING_RATE = 0.01
N_FEATURES = len(X_iris[0])
N_SAMPLES = len(X_iris)
N_PRINTINGS = N_SAMPLES//10
N_QUBITS=math.ceil(np.log2(N_FEATURES)+1) #Nqubits do circuito
weights=np.full(N_FEATURES,1)
QUBITS=[i for i in range(N_QUBITS)]
N_SHOTS=2048
N_ITER=200

## Tratamento do Dataset

In [5]:
def normalize_iqc_ail(data, normalize_col=False, normalize_lin=False):
    if normalize_col:
        scaler = MinMaxScaler() #Normaliza a coluna entre [0,1]
        scaler.fit(data)
        data = scaler.transform(data)
        '''
        Perceba que normalizando apenas a coluna, podemos ter amplitudes dos estados em que a norma do estado não fosse igual a 1. Para resolvermos isso, devemos
        normalizar as linhas entre si

        '''
        data = preprocessing.normalize(data,axis=1,norm='l2')
    if normalize_lin:
        data = preprocessing.normalize(data,axis=1,norm='l2') #Normaliza a linha entre [-1,1]
    return data
    
X_iris_iqc_ail_coluna=normalize_iqc_ail(X_iris, normalize_col=True, normalize_lin=False)
X_iris_iqc_ail_linha=normalize_iqc_ail(X_iris,normalize_col=False,normalize_lin=True)

#### Boxplot IQC:AIL Column Normalized

In [ ]:
fig, ax = plt.subplots()
sns.boxplot(X_iris_iqc_ail_coluna, palette="Set2",ax=ax)
ax.set_xlabel('Features Labels')
ax.set_ylabel('Features Values')
plt.savefig('boxplot_iris_iqc_ail_coluna.svg')

#### Boxplot IQC:AIL Line Normalized

In [ ]:
fig, ax = plt.subplots()
sns.boxplot(X_iris_iqc_ail_linha, palette="Set2",ax=ax)
ax.set_xlabel('Features Labels')
ax.set_ylabel('Features Values')
plt.savefig('boxplot_iris_iqc_ail_linha.svg')

## Função que gera o Quantum Circuit

In [8]:
def blochvector(rho_cog,matriz_pauli_x,matriz_pauli_y,matriz_pauli_z):
    x_bloch = np.trace(matriz_pauli_x@rho_cog.data)
    y_bloch = np.trace(matriz_pauli_y@rho_cog.data)
    z_bloch = np.trace(matriz_pauli_z@rho_cog.data)
    return [x_bloch,y_bloch,z_bloch]
    
#Executar o circuito
def run_qasm_counts(qc, shots=N_SHOTS):
    qc.measure_all()
    qasm_simulator = Aer.get_backend("qasm_simulator")
    job = qasm_simulator.run(qc, shots=shots)
    result = job.result()
    return result.get_counts()

def cirq_iqc_ail(data,contador,w=weights,qubits=QUBITS, N_qubits=N_QUBITS,N_atributos=N_FEATURES,printar_cirq=False):

    X_iris_new=list(data)
    if np.log2(N_atributos)%2!=0 and np.log2(N_atributos)!=1:
        for k in range(2**(N_qubits-1) - N_atributos):
            w=np.append(w,0)
            X_iris_new=np.append(X_iris_new,0)
        sigmaE=np.diag(w)
    else:
        sigmaE=np.diag(w)
    
    #Podíamos inicializar assim pra facilitar as contas
    '''x=np.random.rand(2**N_atributos)
    w=np.random.rand(2**N_atributos)'''

    # IQC:AIL

    qc = QuantumCircuit(N_qubits)

    qc.initialize(X_iris_new, range(1,N_qubits))# Inicializaçao do estado inicial. Poderia ser qualquer estado.
    qc.h(0)
    qc.h(range(1,N_qubits))



    #Montando os sigmas

    matriz_pauli_x=np.array([[0,1],[1,0]]) # Matriz de Pauli x
    matriz_pauli_y=np.array([[0,-1j],[1j,0]]) # Matriz de Pauli y
    matriz_pauli_z=np.array([[1,0],[0,-1]]) # Matriz de Pauli z

    sigmaQ=matriz_pauli_x+matriz_pauli_y+matriz_pauli_z

    

    #Operador Unitário
    U=np.matrix(expMatrix(1j*np.kron(sigmaQ,sigmaE)))

    # qubitstarget = [i for i in range(Ntarget)] - > Desnecessário agora, mas interessante para fazer a generalização
    qc.unitary(U,qubits)
    if contador==0:
        qc.draw("mpl", filename=f'./mpl_complete_U_iris.svg')
    if printar_cirq==True:
        display(qc.draw('mpl')) #display(qc.draw("mpl", filename='./mpl_original.pdf')) #Trocar as chamadas se quiser salvar as imagens dos circuitos

    #qc.decompose().draw(output="mpl", style="clifford")
    qc = transpile(qc, optimization_level=3, basis_gates=["u3", "cx"])
    if dict(qc.count_ops())['u3']<=50 and dict(qc.count_ops())['u3']<=50 and contador%N_PRINTINGS==0:
        qc.draw("mpl", filename=f'./mpl_transpiled{contador}_iris.svg')

    if printar_cirq==True and dict(qc.count_ops())['u3']<=50 and dict(qc.count_ops())['u3']<=50:
        print(dict(qc.count_ops()))
        display(qc.draw('mpl')) #display(qc.draw('mpl', filename='./mpl_transpile.pdf')) #Trocar as chamadas se quiser salvar as imagens dos circuitos

    # Mostrando o vetor de estado 
    sv = Statevector(qc)
    if contador%N_PRINTINGS==0:
        sv.draw("city", filename=f'./state_vector_city{contador}_iris.svg')
        sv.draw("bloch", filename=f'./state_vector_bloch{contador}_iris.svg')
        sv.draw("qsphere", filename=f'./state_vector_qsphere{contador}_iris.svg')
    if printar_cirq==True:
        display(sv.draw("latex"))

    rho_cog = partial_trace(DensityMatrix.from_instruction(qc), qubits[1:])
    if printar_cirq==True:
        print(rho_cog)

    rho_cog_00=rho_cog.data[0,0]
    rho_cog_11=rho_cog.data[1,1]

    if (rho_cog_00 >= rho_cog_11):
        z = 0
    else:
        z = 1
    
    
    counts = run_qasm_counts(qc)
    if contador%N_PRINTINGS==0:
        plot_histogram(counts,filename=f'./histogram_plot_{contador}_iris.svg')
    


    return [z,blochvector(rho_cog,matriz_pauli_x,matriz_pauli_y,matriz_pauli_z)]

## Esfera de Bloch do Circuito

In [9]:
def esfera_bloch_IQC_AIL(X,counter,norma,weights=weights,printar_esf=False):
    point_states=[]
    k_tuple=[]
    z_updt=[]
    for k in range(0,N_SAMPLES):
        k_tuple.append(cirq_iqc_ail(X[k],k,w=weights))
        z_updt.append(k_tuple[0][0])
        point_states.append(k_tuple[0][1])
        k_tuple=[]

    b = qutip.Bloch()
    b.point_default_color=['k']
    b.point_marker=['o']
    b.point_size=[15, 15, 15, 15]
    for k in range(len(point_states)):
        b.add_points(point_states[k])
    b.render()
    if printar_esf==True:
        b.show()

    bb = b.fig
    bb.savefig(f'Bloch_geral_iris{counter}_IQC_AIL_{norma}.svg')
    return z_updt

z1=esfera_bloch_IQC_AIL(X_iris_iqc_ail_coluna,1,'coluna')
z2=esfera_bloch_IQC_AIL(X_iris_iqc_ail_linha,1,'linha')

## Treinamento

### Classificador

In [84]:
def get_U_operator(sigmaQ, sigmaE):
    """
        Makes the exponential matrix of tensor product between sigmaQ and sigmaE and multiplies it by j. 
        
        Equivalent of Equation #15 in the Article.
    """
    return np.matrix(expMatrix(1j*np.kron(sigmaQ, sigmaE)))
def get_p(psi):
    """
        Creates a matrix out of psi and multiply it against its inverse, resulting in a column vector in the form [[alfa]. [beta]].
        
        Does the operation |psi><psi| from Equation #18 or #19 in the Article.
    """
    psi = np.matrix(psi)
    return psi * psi.getH()

def get_negativity(rho, dim):
    """
        Returns the Negativity associated with densitiy matrix rho.
        See definition at: https://en.wikipedia.org/wiki/Negativity_(quantum_mechanics)
        See implementation at: https://toqito.readthedocs.io/en/latest/_autosummary/toqito.state_props.negativity.html
    """
    return state_props.negativity(rho, dim)


def update_weights(weights, y, z, x, p, n):
  """
    Updates the weights. Equation #34 in the Article.
    
    z is the expected classification [0, 1];
    y is the actual classification [0, 1];
    x is the attribute vector;
    p is the probability of the class 1 (0, 1), powered to 2 (p²);
    n is the learning rate.
  """
  # Eq 33
  loss_derivative_on_weight = (1-p)*x

  # Eq 34
  weights = weights-n*(z-y)*loss_derivative_on_weight
  weights[np.isnan(weights)] = 0
  return weights

def iqc_ail_clf(vector_x,w,N,dic_classifier_params={}):
    """
        Applies the a modified version of ICQ classifier using only the math behind the Quantum Classifier described in Interactive Quantum Classifier Inspired by Quantum Open System Theory article. 
        
        It differs from the original ICQ by adding a new component to Sigma Q: sigmaH, which corresponds to a Haddamard's gate. Another difference is that we load the input in the environment instead of having a combination of weights and inputs in sigmaE.

        After doing so, it gets the result of Equation #20 and returns Z as the predicted class and the probability of being the class 1.
        
        Works only for binary classifications, therefore, if the probability of class 0 is needed, it can be 1 - probability of being class 1.

        There are a few possible keys for the dic_classifier_params:
        - sigma_q_params (array) = weights used for calculating sigma_q
        - use_polar_coordinates_on_sigma_q (boolean) = whether to calculate sigma_q using polar coordinates or weighted sum
        - load_inputvector_env_state (boolean) = whether to load input vector on the environment state (True) or on sigma_e (False)
        - operation_for_sigma_e (string) = which operation will be used to combine weights and X for load_inputvector_env_state = False. For now, only "sum" and "mul" are available.
        - calculate_negativity (boolean) = enables the negativity calculation. Check https://en.wikipedia.org/wiki/Negativity_(quantum_mechanics). Uses Toqito implementation: https://toqito.readthedocs.io/en/latest/_autosummary/toqito.state_props.negativity.html
        - ending_hadamard_gate (int) =  adds a Hadamard gate after the U operator
        - use_exponential_on_input (boolean) = does the Euler exponential on the input data after normalizing (if applied)

        To have the original ICQ Classifier, you can have:
        normalize_x = False
        normalize_w = False
        dic_classifier_params["load_inputvector_env_state"] = False
        dic_classifier_params["sigma_q_params"] = [1, 1, 1, 0]

        returns (z, p_cog_new_11_2, output_dict)

        output_dict contains:
        - U_operators = list of used U_operators
        - negativity = negativity associated with that entry
        - entropy = entropy associated with that entry
    """
    #Matrizes de Pauli e a matriz sigma-Q
    matriz_pauli_x=np.array([[0,1],[1,0]]) # Matriz de Pauli x
    matriz_pauli_y=np.array([[0,-1j],[1j,0]]) # Matriz de Pauli y
    matriz_pauli_z=np.array([[1,0],[0,-1]]) # Matriz de Pauli z



    sigmaQ=matriz_pauli_x+matriz_pauli_y+matriz_pauli_z

    sigmaE=np.diag(w)

    # Our first p_cog will be the original one, but will change overtime
    p_cog = np.ones((2,1)) / np.sqrt(2) 
    # Eq #18
    p_cog = get_p(p_cog)

    # We'll update the p_cog for every env we have
    p_cog_new = p_cog
    U_operator = get_U_operator(sigmaQ, sigmaE)

    vector_x_norm = (np.linalg.norm(vector_x) + 1e-16)

    # env = x1/norm(x) |0> + x2/norm(x) |1> .... + xn/norm(x) |n>
    p_env = np.array(vector_x).reshape((N, 1)) / vector_x_norm
    p_env = get_p(p_env)

    # Extracting p_cog and p_env kron
    p_cog_env = np.kron(p_cog_new, p_env)

    # First part of Equation #20 in the Article
    p_out = np.array(U_operator * p_cog_env * U_operator.getH())
    
    # Second part of Equation #20 in the Article
    # For multiple environemnts, this will be our new p_cog
    p_cog_new = np.trace(p_out.reshape([2,N,2,N]), axis1=1, axis2=3)
        
    # As the result is a diagonal matrix, the probability of being class 0 will be on position 0,0
    p_cog_new_00_2 = p_cog_new[0,0]

    # ... and the probability of being class 1 will be on position 1,1
    p_cog_new_11_2 = p_cog_new[1,1]
    if (p_cog_new_00_2 >= p_cog_new_11_2):
        z = 0
    else:
        z = 1

    output_dict = {}
    
    #if "calculate_negativity" in dic_classifier_params and dic_classifier_params["calculate_negativity"]:
    output_dict["negativity"] = get_negativity(p_out, [2, N])

        # with open('C:/Users/Eduardo Barreto/Desktop/Mestrado/icq-studies/experiments/Iris/Entanglement/in_out/evolution_calc.txt', 'a') as file:
        #     string_to_write = "\nvector_x = " + generate_output_matrix_string(vector_x) + ";\n"\
        #                     + "vector_w = " + generate_output_matrix_string(vector_w) + ";\n"\
        #                     + "p_cog_new = " + generate_output_matrix_string(p_cog_new) + ";\n"
        #     file.write(string_to_write)
        #     file.write("\n")
        #     file.write("\n")
        #     file.write("\n")
        #     file.write("--------------------------------------------------------------------------------------------------------")

        # with open('C:/Users/Eduardo Barreto/Desktop/Mestrado/icq-studies/experiments/Iris/Entanglement/in_out/ins_and_outs.txt', 'a') as file:
        #     string_to_write = "\nvector_x = " + generate_output_matrix_string(vector_x) + ";\n"\
        #                     + "vector_w = " + generate_output_matrix_string(vector_w) + ";\n"\
        #                     + "sigmaQ = " + generate_output_matrix_string(sigmaQ) + ";\n"\
        #                     + "sigmaE = " + generate_output_matrix_string(sigmaE) + ";\n"\
        #                     + "p_cog = " + generate_output_matrix_string(p_cog) + ";\n"\
        #                     + "p_env = " + generate_output_matrix_string(p_env) + ";\n"\
        #                     + "p_cog_env = " + generate_output_matrix_string(p_cog_env) + ";\n"\
        #                     + "p_out = " + generate_output_matrix_string(p_out) + ";\n"\
        #                     + "p_cog_new = " + generate_output_matrix_string(p_cog_new) + ";\n"
        #     file.write(string_to_write)
        #     file.write("\n")
        #     file.write("\n")
        #     file.write("\n")
        #     file.write("--------------------------------------------------------------------------------------------------------")

        # with open('C:/Users/Eduardo Barreto/Desktop/Mestrado/icq-studies/experiments/Iris/Entanglement/in_out/negativity.txt', 'a') as file:
        #     string_to_write = "\np_out = " + generate_output_matrix_string(p_out) + ";\n\n - Negativity = " + str(output_dict["negativity"])
        #     file.write(string_to_write)
        #     file.write("\n")
        #     file.write("\n")
        #     file.write("\n")
        #     file.write("--------------------------------------------------------------------------------------------------------")
    
    return z, p_cog_new_11_2, output_dict

### K-folds

In [85]:
def train_10_fold(data, labels, n_features, n_folds=10, learning_rate=LEARNING_RATE, n_iter=N_ITER, print_progress=False):
    """
    Realiza o treinamento 10-fold usando o classificador IQC:AIL.

    Args:
    - data: conjunto de dados (ex.: numpy array)
    - labels: rótulos correspondentes aos dados
    - n_folds: número de folds para a validação cruzada (padrão 10)
    - learning_rate: taxa de aprendizado para a atualização dos pesos
    - n_iter: número de iterações para cada fold
    - print_progress: se True, imprime o progresso e resultados intermediários

    Returns:
    - médias das métricas (acurácia, precisão, recall e F1) para os folds
    """
    kf = KFold(n_splits=n_folds, random_state=42, shuffle=True)
    fold_accuracies = []
    fold_precisions = []
    fold_recalls = []
    fold_f1_scores = []
    weights_list=[]

    # Inicializar pesos (você pode ajustar o tamanho conforme necessário)
    weights = np.full(data.shape[1],0.1)
    weights_list.append(weights)
    if print_progress:
        print(weights)

    negativity_tot=[]
    for fold, (train_index, test_index) in enumerate(kf.split(data)):
        if print_progress:
            print(f"Treinando fold {fold + 1} de {n_folds}...")

        # Dividir dados em treino e teste para este fold
        X_train, X_test = data[train_index], data[test_index]
        y_train, y_test = labels[train_index], labels[test_index]

        # Treinamento
        negativity=[]
        for iteration in range(n_iter):
            for i in range(len(X_train)):
                # Executa a função iqc_ail_clf para obter predição e probabilidade
                x = X_train[i]
                z, p, dic1 = iqc_ail_clf(x,weights,n_features)
                y = y_train[i]
                negativity.append(dic1['negativity'])
                # Atualiza os pesos
                weights = update_weights(weights, y, z, x, p, learning_rate)
                weights_list.append(weights)
        
        y_pred=[]
        for k in range(len(X_test)):
            z, _, _ = iqc_ail_clf(X_test[k],weights,n_features)
            if z==1:
                y_pred.append(y_test[k])
            else:
                y_pred.append(10**6) # Label não existente

        negativity_tot.append(negativity)
        fold_accuracy = accuracy_score(y_test, y_pred)
        fold_precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
        fold_recall = recall_score(y_test, y_pred, average='macro', zero_division=0)
        fold_f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)

        # Armazenar métricas para o fold
        fold_accuracies.append(fold_accuracy)
        fold_precisions.append(fold_precision)
        fold_recalls.append(fold_recall)
        fold_f1_scores.append(fold_f1)

        if print_progress:
            print(f"Fold {fold + 1} - Acurácia: {fold_accuracy:.4f}, Precisão: {fold_precision:.4f}, Recall: {fold_recall:.4f}, F1-score: {fold_f1:.4f}")

    # Calcular médias das métricas entre os folds
    avg_accuracy = np.mean(fold_accuracies)
    avg_precision = np.mean(fold_precisions)
    avg_recall = np.mean(fold_recalls)
    avg_f1_score = np.mean(fold_f1_scores)

    # Exibir resultados finais
    print(f"Média das métricas em {n_folds} folds:")
    print(f"Acurácia: {avg_accuracy:.4f}")
    print(f"Precisão: {avg_precision:.4f}")
    print(f"Recall: {avg_recall:.4f}")
    print(f"F1-score: {avg_f1_score:.4f}")

    return avg_accuracy, avg_precision, avg_recall, avg_f1_score, negativity_tot, weights_list


### Execução

#### Treinamento Coluna

In [ ]:
scores_list_col=[]
scores_list_col.append(train_10_fold(X_iris_iqc_ail_coluna, y_iris, n_features=N_FEATURES, print_progress=True)[0:3])

In [ ]:
scores_list_col

#### Treinamento Linha

In [ ]:
scores_list_lin=[]
scores_list_lin.append(dict(train_10_fold(X_iris_iqc_ail_linha, y_iris, n_features=N_FEATURES, print_progress=True)))

In [ ]:
scores_list_lin